In [1]:
from pathlib import Path
import os

import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

PROJECT_ROOT = Path.cwd()

if not (PROJECT_ROOT / ".env").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

load_dotenv(PROJECT_ROOT / ".env")

required_variables = [
    "DB_USER",
    "DB_PASSWORD",
    "DB_HOST",
    "DB_PORT",
    "DB_NAME",
]

missing_variables = [
    variable
    for variable in required_variables
    if not os.getenv(variable)
]

if missing_variables:
    raise ValueError(
        f"Missing environment variables: {missing_variables}"
    )

database_url = URL.create(
    drivername="mysql+pymysql",
    username=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD"),
    host=os.getenv("DB_HOST"),
    port=int(os.getenv("DB_PORT")),
    database=os.getenv("DB_NAME"),
)

engine = create_engine(
    database_url,
    pool_pre_ping=True,
)

with engine.connect() as connection:
    database_name = connection.execute(
        text("SELECT DATABASE()")
    ).scalar()

    database_version = connection.execute(
        text("SELECT VERSION()")
    ).scalar()

print(f"Connected database: {database_name}")
print(f"MariaDB version: {database_version}")

Connected database: ecommerce_product_intelligence
MariaDB version: 12.3.3-MariaDB


In [2]:
monthly_kpi_view_sql = """
CREATE OR REPLACE VIEW vw_monthly_kpis AS
SELECT
    DATE_FORMAT(purchase_date, '%Y-%m-01') AS purchase_month,
    COUNT(*) AS total_orders,
    SUM(order_status = 'delivered') AS delivered_orders,

    ROUND(
        SUM(
            CASE
                WHEN order_status = 'delivered'
                THEN COALESCE(total_order_value, 0)
                ELSE 0
            END
        ),
        2
    ) AS delivered_revenue,

    ROUND(
        AVG(
            CASE
                WHEN order_status = 'delivered'
                THEN total_order_value
            END
        ),
        2
    ) AS average_order_value,

    ROUND(
        AVG(
            CASE
                WHEN order_status = 'delivered'
                THEN actual_delivery_days
            END
        ),
        2
    ) AS average_delivery_days,

    ROUND(
        100 * AVG(
            CASE
                WHEN order_status = 'delivered'
                     AND is_late_delivery IS NOT NULL
                THEN is_late_delivery
            END
        ),
        2
    ) AS late_delivery_rate,

    ROUND(
        AVG(
            CASE
                WHEN order_status = 'delivered'
                THEN latest_review_score
            END
        ),
        2
    ) AS average_review_score

FROM fact_orders
WHERE purchase_date IS NOT NULL
GROUP BY DATE_FORMAT(purchase_date, '%Y-%m-01')
"""

with engine.begin() as connection:
    connection.execute(text(monthly_kpi_view_sql))

SQL_DIR = PROJECT_ROOT / "sql"
SQL_DIR.mkdir(parents=True, exist_ok=True)

(SQL_DIR / "01_monthly_kpis_view.sql").write_text(
    monthly_kpi_view_sql.strip() + ";\n",
    encoding="utf-8",
)

with engine.connect() as connection:
    monthly_kpis = pd.read_sql(
        text("""
            SELECT *
            FROM vw_monthly_kpis
            ORDER BY purchase_month
        """),
        connection,
    )

print(f"Monthly periods returned: {len(monthly_kpis)}")
monthly_kpis

Monthly periods returned: 25


,purchase_month,total_orders,delivered_orders,delivered_revenue,average_order_value,average_delivery_days,late_delivery_rate,average_review_score
0,2016-09-01,4,1.0,143.46,143.46,54.81,100.00,1.00
1,2016-10-01,324,265.0,46490.66,175.44,19.60,1.13,4.01
2,2016-12-01,1,1.0,19.62,19.62,4.69,0.00,5.00
3,2017-01-01,800,750.0,127482.37,169.98,12.65,3.07,4.20
4,2017-02-01,1780,1653.0,271239.32,164.09,13.17,3.21,4.20
5,2017-03-01,2682,2546.0,414330.95,162.74,12.95,5.58,4.19
6,2017-04-01,2404,2303.0,390812.40,169.70,14.92,7.86,4.14
7,2017-05-01,3700,3546.0,566851.40,159.86,11.32,3.61,4.24
8,2017-06-01,3245,3135.0,490050.37,156.32,12.01,3.86,4.22
9,2017-07-01,4026,3872.0,566299.08,146.25,11.59,3.43,4.26


In [3]:
category_view_sql = """
CREATE OR REPLACE VIEW vw_category_performance AS
SELECT
    order_category.product_category,
    SUM(order_category.units_sold) AS units_sold,
    COUNT(*) AS order_count,

    ROUND(
        SUM(order_category.product_revenue),
        2
    ) AS product_revenue,

    ROUND(
        SUM(order_category.freight_value),
        2
    ) AS freight_value,

    ROUND(
        SUM(order_category.order_category_value),
        2
    ) AS total_value,

    ROUND(
        AVG(order_category.order_category_value),
        2
    ) AS average_order_category_value,

    ROUND(
        100 * SUM(order_category.freight_value)
        / NULLIF(SUM(order_category.order_category_value), 0),
        2
    ) AS freight_percentage,

    ROUND(
        AVG(order_category.actual_delivery_days),
        2
    ) AS average_delivery_days,

    ROUND(
        100 * AVG(
            CASE
                WHEN order_category.is_late_delivery IS NOT NULL
                THEN order_category.is_late_delivery
            END
        ),
        2
    ) AS late_delivery_rate,

    ROUND(
        AVG(order_category.review_score),
        2
    ) AS average_review_score,

    ROUND(
        100 * AVG(
            CASE
                WHEN order_category.review_score IS NULL THEN NULL
                WHEN order_category.review_score <= 2 THEN 1
                ELSE 0
            END
        ),
        2
    ) AS low_review_rate

FROM (
    SELECT
        order_id,
        product_category,
        COUNT(*) AS units_sold,
        SUM(price) AS product_revenue,
        SUM(freight_value) AS freight_value,
        SUM(item_total_value) AS order_category_value,
        MAX(actual_delivery_days) AS actual_delivery_days,
        MAX(is_late_delivery) AS is_late_delivery,
        MAX(latest_review_score) AS review_score
    FROM fact_order_items
    WHERE order_status = 'delivered'
      AND product_category IS NOT NULL
    GROUP BY
        order_id,
        product_category
) AS order_category

GROUP BY order_category.product_category
"""

with engine.begin() as connection:
    connection.execute(text(category_view_sql))

(SQL_DIR / "02_category_performance_view.sql").write_text(
    category_view_sql.strip() + ";\n",
    encoding="utf-8",
)

with engine.connect() as connection:
    category_performance = pd.read_sql(
        text("""
            SELECT *
            FROM vw_category_performance
            WHERE order_count >= 100
            ORDER BY total_value DESC
            LIMIT 15
        """),
        connection,
    )

print(
    f"Categories represented in view: "
    f"{pd.read_sql('SELECT COUNT(*) AS count FROM vw_category_performance', engine).iloc[0, 0]}"
)

category_performance

Categories represented in view: 74


,product_category,units_sold,order_count,product_revenue,freight_value,total_value,average_order_category_value,freight_percentage,average_delivery_days,late_delivery_rate,average_review_score,low_review_rate
0,health_beauty,9465.0,8647,1233131.72,178957.81,1412089.53,163.30,12.67,12.08,8.96,4.23,11.45
1,watches_gifts,5859.0,5495,1166176.98,98156.14,1264333.12,230.09,7.76,12.81,8.52,4.12,13.70
2,bed_bath_table,10953.0,9272,1023434.76,201774.50,1225209.26,132.14,16.47,12.99,8.75,4.00,16.03
3,sports_leisure,8431.0,7530,954852.55,163404.36,1118256.91,148.51,14.61,12.24,7.76,4.23,11.38
4,computers_accessories,7644.0,6530,888724.61,143999.16,1032723.77,158.15,13.94,13.17,7.70,4.08,14.56
5,furniture_decor,8160.0,6307,711927.69,168402.23,880329.92,139.58,19.13,13.07,8.48,4.06,15.35
6,housewares,6795.0,5743,615628.69,142763.56,758392.25,132.06,18.82,11.09,6.95,4.19,11.82
7,cool_stuff,3718.0,3559,610204.10,81476.79,691680.89,194.35,11.78,12.38,6.83,4.22,11.27
8,auto,4140.0,3810,578966.65,90488.10,669454.75,175.71,13.52,12.33,8.61,4.15,13.00
9,garden_tools,4268.0,3448,470495.28,96650.40,567145.68,164.49,17.04,13.63,7.95,4.18,12.25


## 3. How does shipping distance affect cost and customer experience?

Items are aggregated to the order-seller level before analysis so multi-item shipments do not disproportionately influence delivery and review metrics.

In [4]:
distance_view_sql = """
CREATE OR REPLACE VIEW vw_distance_performance AS
SELECT
    CASE
        WHEN order_seller.distance_km < 100 THEN '0–99 km'
        WHEN order_seller.distance_km < 300 THEN '100–299 km'
        WHEN order_seller.distance_km < 600 THEN '300–599 km'
        WHEN order_seller.distance_km < 1000 THEN '600–999 km'
        WHEN order_seller.distance_km < 1500 THEN '1,000–1,499 km'
        ELSE '1,500+ km'
    END AS distance_band,

    CASE
        WHEN order_seller.distance_km < 100 THEN 1
        WHEN order_seller.distance_km < 300 THEN 2
        WHEN order_seller.distance_km < 600 THEN 3
        WHEN order_seller.distance_km < 1000 THEN 4
        WHEN order_seller.distance_km < 1500 THEN 5
        ELSE 6
    END AS distance_band_order,

    COUNT(*) AS shipment_count,
    COUNT(DISTINCT order_seller.order_id) AS order_count,
    COUNT(DISTINCT order_seller.seller_id) AS seller_count,

    ROUND(
        AVG(order_seller.distance_km),
        2
    ) AS average_distance_km,

    ROUND(
        SUM(order_seller.shipment_value),
        2
    ) AS total_value,

    ROUND(
        AVG(order_seller.freight_value),
        2
    ) AS average_freight_value,

    ROUND(
        100 * SUM(order_seller.freight_value)
        / NULLIF(SUM(order_seller.shipment_value), 0),
        2
    ) AS freight_percentage,

    ROUND(
        AVG(order_seller.actual_delivery_days),
        2
    ) AS average_delivery_days,

    ROUND(
        100 * AVG(order_seller.is_late_delivery),
        2
    ) AS late_delivery_rate,

    ROUND(
        AVG(order_seller.review_score),
        2
    ) AS average_review_score,

    ROUND(
        100 * AVG(
            CASE
                WHEN order_seller.review_score IS NULL THEN NULL
                WHEN order_seller.review_score <= 2 THEN 1
                ELSE 0
            END
        ),
        2
    ) AS low_review_rate

FROM (
    SELECT
        order_id,
        seller_id,
        MAX(seller_customer_distance_km) AS distance_km,
        SUM(item_total_value) AS shipment_value,
        SUM(freight_value) AS freight_value,
        MAX(actual_delivery_days) AS actual_delivery_days,
        MAX(is_late_delivery) AS is_late_delivery,
        MAX(latest_review_score) AS review_score
    FROM fact_order_items
    WHERE order_status = 'delivered'
      AND seller_customer_distance_km IS NOT NULL
    GROUP BY
        order_id,
        seller_id
) AS order_seller

GROUP BY
    distance_band,
    distance_band_order
"""

with engine.begin() as connection:
    connection.execute(text(distance_view_sql))

(SQL_DIR / "03_distance_performance_view.sql").write_text(
    distance_view_sql.strip() + ";\n",
    encoding="utf-8",
)

with engine.connect() as connection:
    distance_performance = pd.read_sql(
        text("""
            SELECT *
            FROM vw_distance_performance
            ORDER BY distance_band_order
        """),
        connection,
    )

distance_performance

,distance_band,distance_band_order,shipment_count,order_count,seller_count,average_distance_km,total_value,average_freight_value,freight_percentage,average_delivery_days,late_delivery_rate,average_review_score,low_review_rate
0,0–99 km,1,18092,17986,1614,40.05,2277481.38,13.29,10.56,6.49,6.34,4.26,10.87
1,100–299 km,2,13578,13488,1568,200.32,2007351.08,18.48,12.50,9.95,6.33,4.23,11.24
2,300–599 km,3,31961,31676,2206,430.03,4920307.95,22.05,14.32,12.58,7.68,4.11,13.95
3,600–999 km,4,18178,18026,1922,784.84,2943786.32,24.46,15.11,14.65,8.36,4.10,13.83
4,"1,000–1,499 km",5,6672,6620,1341,1234.06,1300372.74,29.34,15.05,17.18,9.98,4.03,15.53
5,"1,500+ km",6,8851,8791,1400,2113.08,1897372.41,39.55,18.45,20.49,12.98,3.99,16.28


In [5]:
distance_performance[
    [
        "distance_band",
        "shipment_count",
        "average_freight_value",
        "freight_percentage",
        "average_delivery_days",
        "late_delivery_rate",
        "average_review_score",
        "low_review_rate",
    ]
]

,distance_band,shipment_count,average_freight_value,freight_percentage,average_delivery_days,late_delivery_rate,average_review_score,low_review_rate
0,0–99 km,18092,13.29,10.56,6.49,6.34,4.26,10.87
1,100–299 km,13578,18.48,12.50,9.95,6.33,4.23,11.24
2,300–599 km,31961,22.05,14.32,12.58,7.68,4.11,13.95
3,600–999 km,18178,24.46,15.11,14.65,8.36,4.10,13.83
4,"1,000–1,499 km",6672,29.34,15.05,17.18,9.98,4.03,15.53
5,"1,500+ km",8851,39.55,18.45,20.49,12.98,3.99,16.28


### Finding

Long-distance fulfillment represents a measurable operational risk. Compared with shipments under 100 km, shipments traveling at least 1,500 km incurred nearly three times the average freight cost, required over three times as many delivery days, and experienced more than double the late-delivery rate. Average review scores also declined from 4.26 to 3.99.

## 4. How strongly does delivery timing affect customer satisfaction?

Delivered orders are segmented by how early or late they arrived relative to the promised delivery date.

In [6]:
delivery_experience_view_sql = """
CREATE OR REPLACE VIEW vw_delivery_experience AS
SELECT
    CASE
        WHEN days_from_estimate <= -8 THEN '8+ days early'
        WHEN days_from_estimate <= -1 THEN '1–7 days early'
        WHEN days_from_estimate <= 1 THEN 'On schedule'
        WHEN days_from_estimate <= 3 THEN '1–3 days late'
        WHEN days_from_estimate <= 7 THEN '4–7 days late'
        ELSE '8+ days late'
    END AS delivery_timing,

    CASE
        WHEN days_from_estimate <= -8 THEN 1
        WHEN days_from_estimate <= -1 THEN 2
        WHEN days_from_estimate <= 1 THEN 3
        WHEN days_from_estimate <= 3 THEN 4
        WHEN days_from_estimate <= 7 THEN 5
        ELSE 6
    END AS delivery_timing_order,

    COUNT(*) AS delivered_orders,

    ROUND(
        AVG(actual_delivery_days),
        2
    ) AS average_delivery_days,

    ROUND(
        AVG(days_from_estimate),
        2
    ) AS average_days_from_estimate,

    ROUND(
        AVG(latest_review_score),
        2
    ) AS average_review_score,

    ROUND(
        100 * AVG(
            CASE
                WHEN latest_review_score IS NULL THEN NULL
                WHEN latest_review_score <= 2 THEN 1
                ELSE 0
            END
        ),
        2
    ) AS low_review_rate,

    ROUND(
        100 * AVG(
            CASE
                WHEN latest_review_score IS NULL THEN NULL
                WHEN latest_review_score = 5 THEN 1
                ELSE 0
            END
        ),
        2
    ) AS five_star_review_rate,

    ROUND(
        AVG(total_order_value),
        2
    ) AS average_order_value

FROM fact_orders
WHERE order_status = 'delivered'
  AND days_from_estimate IS NOT NULL

GROUP BY
    delivery_timing,
    delivery_timing_order
"""

with engine.begin() as connection:
    connection.execute(text(delivery_experience_view_sql))

(SQL_DIR / "04_delivery_experience_view.sql").write_text(
    delivery_experience_view_sql.strip() + ";\n",
    encoding="utf-8",
)

with engine.connect() as connection:
    delivery_experience = pd.read_sql(
        text("""
            SELECT *
            FROM vw_delivery_experience
            ORDER BY delivery_timing_order
        """),
        connection,
    )

delivery_experience

,delivery_timing,delivery_timing_order,delivered_orders,average_delivery_days,average_days_from_estimate,average_review_score,low_review_rate,five_star_review_rate,average_order_value
0,8+ days early,1,66475,10.13,-15.73,4.32,8.91,63.84,162.20
1,1–7 days early,2,20707,12.89,-5.16,4.22,10.10,58.41,148.34
2,On schedule,3,2754,18.02,0.17,4.10,11.68,53.22,150.21
3,1–3 days late,4,1370,22.82,2.07,3.51,25.44,36.21,159.38
4,4–7 days late,5,1819,26.68,5.12,2.32,61.31,18.10,178.64
5,8+ days late,6,3345,42.46,18.45,1.73,78.44,7.47,181.64


In [7]:
delivery_experience_view_sql = """
CREATE OR REPLACE VIEW vw_delivery_experience AS
SELECT
    CASE
        WHEN days_from_estimate <= -8 THEN '8+ days early'
        WHEN days_from_estimate <= -1 THEN '1–7 days early'
        WHEN days_from_estimate <= 1 THEN 'On schedule'
        WHEN days_from_estimate <= 3 THEN '1–3 days late'
        WHEN days_from_estimate <= 7 THEN '4–7 days late'
        ELSE '8+ days late'
    END AS delivery_timing,

    CASE
        WHEN days_from_estimate <= -8 THEN 1
        WHEN days_from_estimate <= -1 THEN 2
        WHEN days_from_estimate <= 1 THEN 3
        WHEN days_from_estimate <= 3 THEN 4
        WHEN days_from_estimate <= 7 THEN 5
        ELSE 6
    END AS delivery_timing_order,

    COUNT(*) AS delivered_orders,

    ROUND(
        AVG(actual_delivery_days),
        2
    ) AS average_delivery_days,

    ROUND(
        AVG(days_from_estimate),
        2
    ) AS average_days_from_estimate,

    ROUND(
        AVG(latest_review_score),
        2
    ) AS average_review_score,

    ROUND(
        100 * AVG(
            CASE
                WHEN latest_review_score IS NULL THEN NULL
                WHEN latest_review_score <= 2 THEN 1
                ELSE 0
            END
        ),
        2
    ) AS low_review_rate,

    ROUND(
        100 * AVG(
            CASE
                WHEN latest_review_score IS NULL THEN NULL
                WHEN latest_review_score = 5 THEN 1
                ELSE 0
            END
        ),
        2
    ) AS five_star_review_rate,

    ROUND(
        AVG(total_order_value),
        2
    ) AS average_order_value

FROM fact_orders
WHERE order_status = 'delivered'
  AND days_from_estimate IS NOT NULL

GROUP BY
    delivery_timing,
    delivery_timing_order
"""

with engine.begin() as connection:
    connection.execute(text(delivery_experience_view_sql))

(SQL_DIR / "04_delivery_experience_view.sql").write_text(
    delivery_experience_view_sql.strip() + ";\n",
    encoding="utf-8",
)

with engine.connect() as connection:
    delivery_experience = pd.read_sql(
        text("""
            SELECT *
            FROM vw_delivery_experience
            ORDER BY delivery_timing_order
        """),
        connection,
    )

delivery_experience

,delivery_timing,delivery_timing_order,delivered_orders,average_delivery_days,average_days_from_estimate,average_review_score,low_review_rate,five_star_review_rate,average_order_value
0,8+ days early,1,66475,10.13,-15.73,4.32,8.91,63.84,162.20
1,1–7 days early,2,20707,12.89,-5.16,4.22,10.10,58.41,148.34
2,On schedule,3,2754,18.02,0.17,4.10,11.68,53.22,150.21
3,1–3 days late,4,1370,22.82,2.07,3.51,25.44,36.21,159.38
4,4–7 days late,5,1819,26.68,5.12,2.32,61.31,18.10,178.64
5,8+ days late,6,3345,42.46,18.45,1.73,78.44,7.47,181.64


## 5. How do repeat customers differ from one-time customers?

Customers are classified using their complete purchase history. The analysis compares retention, revenue, order value, delivery reliability, and satisfaction.

In [8]:
customer_retention_view_sql = """
CREATE OR REPLACE VIEW vw_customer_retention AS
SELECT
    CASE
        WHEN customer_summary.total_orders = 1
        THEN 'One-time customers'
        ELSE 'Repeat customers'
    END AS customer_segment,

    COUNT(*) AS customer_count,
    SUM(customer_summary.total_orders) AS total_orders,

    ROUND(
        AVG(customer_summary.total_orders),
        2
    ) AS average_orders_per_customer,

    SUM(
        customer_summary.delivered_orders
    ) AS delivered_orders,

    ROUND(
        SUM(customer_summary.delivered_revenue),
        2
    ) AS delivered_revenue,

    ROUND(
        AVG(customer_summary.delivered_revenue),
        2
    ) AS average_customer_revenue,

    ROUND(
        SUM(customer_summary.delivered_revenue)
        / NULLIF(SUM(customer_summary.delivered_orders), 0),
        2
    ) AS average_order_value,

    ROUND(
        100 * SUM(customer_summary.late_delivery_count)
        / NULLIF(SUM(customer_summary.evaluated_deliveries), 0),
        2
    ) AS late_delivery_rate,

    ROUND(
        SUM(customer_summary.review_score_total)
        / NULLIF(SUM(customer_summary.reviewed_orders), 0),
        2
    ) AS average_review_score,

    ROUND(
        AVG(
            DATEDIFF(
                customer_summary.latest_purchase_date,
                customer_summary.first_purchase_date
            )
        ),
        2
    ) AS average_customer_span_days

FROM (
    SELECT
        customer_unique_id,
        COUNT(*) AS total_orders,

        SUM(
            order_status = 'delivered'
        ) AS delivered_orders,

        SUM(
            CASE
                WHEN order_status = 'delivered'
                THEN COALESCE(total_order_value, 0)
                ELSE 0
            END
        ) AS delivered_revenue,

        SUM(
            CASE
                WHEN order_status = 'delivered'
                     AND is_late_delivery IS NOT NULL
                THEN is_late_delivery
                ELSE 0
            END
        ) AS late_delivery_count,

        SUM(
            CASE
                WHEN order_status = 'delivered'
                     AND is_late_delivery IS NOT NULL
                THEN 1
                ELSE 0
            END
        ) AS evaluated_deliveries,

        SUM(
            CASE
                WHEN order_status = 'delivered'
                     AND latest_review_score IS NOT NULL
                THEN latest_review_score
                ELSE 0
            END
        ) AS review_score_total,

        SUM(
            CASE
                WHEN order_status = 'delivered'
                     AND latest_review_score IS NOT NULL
                THEN 1
                ELSE 0
            END
        ) AS reviewed_orders,

        MIN(purchase_date) AS first_purchase_date,
        MAX(purchase_date) AS latest_purchase_date

    FROM fact_orders
    WHERE customer_unique_id IS NOT NULL
    GROUP BY customer_unique_id
) AS customer_summary

GROUP BY customer_segment
"""

with engine.begin() as connection:
    connection.execute(text(customer_retention_view_sql))

(SQL_DIR / "05_customer_retention_view.sql").write_text(
    customer_retention_view_sql.strip() + ";\n",
    encoding="utf-8",
)

with engine.connect() as connection:
    customer_retention = pd.read_sql(
        text("""
            SELECT *
            FROM vw_customer_retention
            ORDER BY average_orders_per_customer
        """),
        connection,
    )

customer_retention

,customer_segment,customer_count,total_orders,average_orders_per_customer,delivered_orders,delivered_revenue,average_customer_revenue,average_order_value,late_delivery_rate,average_review_score,average_customer_span_days
0,One-time customers,93099,93099.0,1.00,90379.0,14519216.15,155.95,160.65,8.19,4.15,0.00
1,Repeat customers,2997,6342.0,2.12,6099.0,900557.60,300.49,147.66,7.03,4.20,87.31


## 6. Which high-volume sellers present the greatest operational risk?

Seller performance is evaluated at the order-seller shipment level using sales value, freight burden, shipping distance, delivery reliability, and customer reviews.

In [9]:
seller_performance_view_sql = """
CREATE OR REPLACE VIEW vw_seller_performance AS
SELECT
    order_seller.seller_id,
    MAX(order_seller.seller_city) AS seller_city,
    MAX(order_seller.seller_state) AS seller_state,

    COUNT(*) AS shipment_count,
    COUNT(DISTINCT order_seller.order_id) AS order_count,

    ROUND(
        SUM(order_seller.product_revenue),
        2
    ) AS product_revenue,

    ROUND(
        SUM(order_seller.freight_value),
        2
    ) AS freight_value,

    ROUND(
        SUM(order_seller.shipment_value),
        2
    ) AS total_value,

    ROUND(
        AVG(order_seller.shipment_value),
        2
    ) AS average_shipment_value,

    ROUND(
        100 * SUM(order_seller.freight_value)
        / NULLIF(SUM(order_seller.shipment_value), 0),
        2
    ) AS freight_percentage,

    ROUND(
        AVG(order_seller.distance_km),
        2
    ) AS average_distance_km,

    ROUND(
        AVG(order_seller.actual_delivery_days),
        2
    ) AS average_delivery_days,

    ROUND(
        100 * AVG(order_seller.is_late_delivery),
        2
    ) AS late_delivery_rate,

    ROUND(
        AVG(order_seller.review_score),
        2
    ) AS average_review_score,

    ROUND(
        100 * AVG(
            CASE
                WHEN order_seller.review_score IS NULL THEN NULL
                WHEN order_seller.review_score <= 2 THEN 1
                ELSE 0
            END
        ),
        2
    ) AS low_review_rate

FROM (
    SELECT
        order_id,
        seller_id,
        MAX(seller_city) AS seller_city,
        MAX(seller_state) AS seller_state,
        SUM(price) AS product_revenue,
        SUM(freight_value) AS freight_value,
        SUM(item_total_value) AS shipment_value,
        MAX(seller_customer_distance_km) AS distance_km,
        MAX(actual_delivery_days) AS actual_delivery_days,
        MAX(is_late_delivery) AS is_late_delivery,
        MAX(latest_review_score) AS review_score
    FROM fact_order_items
    WHERE order_status = 'delivered'
    GROUP BY
        order_id,
        seller_id
) AS order_seller

GROUP BY order_seller.seller_id
"""

with engine.begin() as connection:
    connection.execute(text(seller_performance_view_sql))

(SQL_DIR / "06_seller_performance_view.sql").write_text(
    seller_performance_view_sql.strip() + ";\n",
    encoding="utf-8",
)

with engine.connect() as connection:
    high_volume_seller_risks = pd.read_sql(
        text("""
            SELECT *
            FROM vw_seller_performance
            WHERE shipment_count >= 100
            ORDER BY
                late_delivery_rate DESC,
                low_review_rate DESC,
                total_value DESC
            LIMIT 15
        """),
        connection,
    )

high_volume_seller_risks

,seller_id,seller_city,seller_state,shipment_count,order_count,product_revenue,freight_value,total_value,average_shipment_value,freight_percentage,average_distance_km,average_delivery_days,late_delivery_rate,average_review_score,low_review_rate
0,06a2c3af7b3aee5d69171b0e14f0ee87,sao luis,MA,389,389,36097.98,12071.52,48169.50,123.83,25.06,1944.65,17.73,23.14,4.01,16.10
1,1ca7077d890b907f89be8c954a02686a,santana de parnaiba,SP,108,108,12474.64,1522.01,13996.65,129.60,10.87,390.85,15.54,22.22,2.39,59.81
2,88460e8ebdecbfecb5f9601833981930,maringa,PR,246,246,31320.40,4938.19,36258.59,147.39,13.62,664.69,17.80,19.51,3.43,30.61
3,e5a3438891c0bfdb9394643f95273d8e,limeira,SP,216,216,7549.96,3280.74,10830.70,50.14,30.29,495.71,15.66,18.52,3.99,17.67
4,cd68562d3f44870c08922d380acae552,ribeirao preto,SP,122,122,18119.70,2160.16,20279.86,166.23,10.65,694.44,16.26,18.03,4.02,16.53
5,8160255418d5aaa7dbdc9f4c64ebda44,ibitinga,SP,380,380,46368.39,7727.29,54095.68,142.36,14.28,573.90,16.25,16.58,3.95,17.02
6,2c9e548be18521d1c43cde1c582c6de8,mogi das cruzes,SP,124,124,6013.56,2456.53,8470.09,68.31,29.00,369.53,13.68,16.13,4.13,13.11
7,dd7ddc04e1b6c2c614352b383efe2d36,sao paulo,SP,121,121,9148.61,2876.57,12025.18,99.38,23.92,422.08,15.02,15.70,3.88,18.33
8,f7ba60f8c3f99e7ee4042fdef03b70c4,sao bernardo do campo,SP,218,218,68070.00,4317.94,72387.94,332.05,5.96,973.89,13.41,15.60,4.22,10.55
9,431af27f296bc6519d890aa5a05fdb11,ribeirao preto,SP,116,116,12974.60,2128.45,15103.05,130.20,14.09,559.62,17.23,15.52,3.80,23.48


## 7. Which geographic markets generate the most value and operational risk?

State-level performance compares demand, revenue, retention, fulfillment speed, late-delivery rates, and customer satisfaction.

In [10]:
state_performance_view_sql = """
CREATE OR REPLACE VIEW vw_state_performance AS
SELECT
    customer_state,
    'Brazil' AS country,

    COUNT(*) AS total_orders,
    COUNT(DISTINCT customer_unique_id) AS unique_customers,

    SUM(
        order_status = 'delivered'
    ) AS delivered_orders,

    ROUND(
        SUM(
            CASE
                WHEN order_status = 'delivered'
                THEN COALESCE(total_order_value, 0)
                ELSE 0
            END
        ),
        2
    ) AS delivered_revenue,

    ROUND(
        AVG(
            CASE
                WHEN order_status = 'delivered'
                THEN total_order_value
            END
        ),
        2
    ) AS average_order_value,

    ROUND(
        AVG(
            CASE
                WHEN order_status = 'delivered'
                THEN actual_delivery_days
            END
        ),
        2
    ) AS average_delivery_days,

    ROUND(
        100 * AVG(
            CASE
                WHEN order_status = 'delivered'
                     AND is_late_delivery IS NOT NULL
                THEN is_late_delivery
            END
        ),
        2
    ) AS late_delivery_rate,

    ROUND(
        AVG(
            CASE
                WHEN order_status = 'delivered'
                THEN latest_review_score
            END
        ),
        2
    ) AS average_review_score,

    ROUND(
        100 * AVG(is_repeat_customer_order),
        2
    ) AS repeat_order_rate,

    ROUND(
        AVG(customer_latitude),
        6
    ) AS average_latitude,

    ROUND(
        AVG(customer_longitude),
        6
    ) AS average_longitude

FROM fact_orders
WHERE customer_state IS NOT NULL
GROUP BY customer_state
"""

with engine.begin() as connection:
    connection.execute(text(state_performance_view_sql))

(SQL_DIR / "07_state_performance_view.sql").write_text(
    state_performance_view_sql.strip() + ";\n",
    encoding="utf-8",
)

with engine.connect() as connection:
    state_performance = pd.read_sql(
        text("""
            SELECT *
            FROM vw_state_performance
            ORDER BY delivered_revenue DESC
        """),
        connection,
    )

print(f"States represented: {len(state_performance)}")
state_performance

States represented: 27


,customer_state,country,total_orders,unique_customers,delivered_orders,delivered_revenue,average_order_value,average_delivery_days,late_delivery_rate,average_review_score,repeat_order_rate,average_latitude,average_longitude
0,SP,Brazil,41746,40302,40501.0,5769703.15,142.46,8.76,5.89,4.25,3.49,-23.167072,-47.048357
1,RJ,Brazil,12852,12384,12350.0,2055401.57,166.43,15.31,13.47,3.97,3.68,-22.760707,-43.176336
2,MG,Brazil,11635,11259,11354.0,1818891.67,160.20,12.01,5.61,4.19,3.26,-19.921527,-44.437928
3,RS,Brazil,5466,5277,5345.0,861472.79,161.17,15.30,7.15,4.18,3.48,-29.715133,-51.944664
4,PR,Brazil,5045,4882,4923.0,781708.80,158.79,11.99,5.00,4.24,3.25,-24.802047,-50.808006
5,SC,Brazil,3637,3534,3546.0,595127.78,167.83,14.95,9.76,4.13,2.97,-27.242880,-49.558934
6,BA,Brazil,3380,3277,3256.0,591137.81,181.55,19.34,14.04,3.93,3.05,-13.038127,-39.456456
7,DF,Brazil,2140,2075,2080.0,346123.35,166.41,12.97,7.07,4.13,3.08,-15.810267,-47.976556
8,GO,Brazil,2020,1952,1957.0,334212.35,170.78,15.61,8.18,4.11,3.42,-16.607326,-49.334738
9,ES,Brazil,2033,1964,1995.0,317657.93,159.23,15.79,12.23,4.08,3.44,-20.155214,-40.474246


## 8. Executive Summary and Dashboard Exports

A one-row executive KPI view is created, and all analytical SQL views are exported as compact CSV files for dashboard development.

In [11]:
executive_summary_view_sql = """
CREATE OR REPLACE VIEW vw_executive_summary AS
SELECT
    COUNT(*) AS total_orders,
    COUNT(DISTINCT customer_unique_id) AS unique_customers,

    SUM(
        order_status = 'delivered'
    ) AS delivered_orders,

    ROUND(
        100 * SUM(order_status = 'delivered') / COUNT(*),
        2
    ) AS delivery_completion_rate,

    ROUND(
        SUM(
            CASE
                WHEN order_status = 'delivered'
                THEN COALESCE(total_order_value, 0)
                ELSE 0
            END
        ),
        2
    ) AS delivered_revenue,

    ROUND(
        AVG(
            CASE
                WHEN order_status = 'delivered'
                THEN total_order_value
            END
        ),
        2
    ) AS average_order_value,

    ROUND(
        AVG(
            CASE
                WHEN order_status = 'delivered'
                THEN actual_delivery_days
            END
        ),
        2
    ) AS average_delivery_days,

    ROUND(
        100 * AVG(
            CASE
                WHEN order_status = 'delivered'
                     AND is_late_delivery IS NOT NULL
                THEN is_late_delivery
            END
        ),
        2
    ) AS late_delivery_rate,

    ROUND(
        AVG(
            CASE
                WHEN order_status = 'delivered'
                THEN latest_review_score
            END
        ),
        2
    ) AS average_review_score,

    SUM(is_repeat_customer_order) AS repeat_orders,

    ROUND(
        100 * AVG(is_repeat_customer_order),
        2
    ) AS repeat_order_rate

FROM fact_orders
"""

with engine.begin() as connection:
    connection.execute(text(executive_summary_view_sql))

(SQL_DIR / "08_executive_summary_view.sql").write_text(
    executive_summary_view_sql.strip() + ";\n",
    encoding="utf-8",
)

1577

In [12]:
DASHBOARD_DATA_DIR = PROJECT_ROOT / "dashboard" / "data"
DASHBOARD_DATA_DIR.mkdir(parents=True, exist_ok=True)

dashboard_exports = {
    "executive_summary.csv": """
        SELECT * FROM vw_executive_summary
    """,
    "monthly_kpis.csv": """
        SELECT * FROM vw_monthly_kpis
        ORDER BY purchase_month
    """,
    "category_performance.csv": """
        SELECT * FROM vw_category_performance
        ORDER BY total_value DESC
    """,
    "distance_performance.csv": """
        SELECT * FROM vw_distance_performance
        ORDER BY distance_band_order
    """,
    "delivery_experience.csv": """
        SELECT * FROM vw_delivery_experience
        ORDER BY delivery_timing_order
    """,
    "customer_retention.csv": """
        SELECT * FROM vw_customer_retention
        ORDER BY average_orders_per_customer
    """,
    "seller_performance.csv": """
        SELECT * FROM vw_seller_performance
        ORDER BY total_value DESC
    """,
    "state_performance.csv": """
        SELECT * FROM vw_state_performance
        ORDER BY delivered_revenue DESC
    """,
}

export_records = []

with engine.connect() as connection:
    for filename, query in dashboard_exports.items():
        dataframe = pd.read_sql(
            text(query),
            connection,
        )

        output_path = DASHBOARD_DATA_DIR / filename
        dataframe.to_csv(output_path, index=False)

        export_records.append({
            "file": filename,
            "rows": len(dataframe),
            "columns": len(dataframe.columns),
            "size_kb": round(
                output_path.stat().st_size / 1_000,
                2,
            ),
        })

dashboard_export_summary = pd.DataFrame(export_records)

assert len(dashboard_export_summary) == 8
assert (dashboard_export_summary["rows"] > 0).all()

print("Exported 8 dashboard-ready datasets.")
dashboard_export_summary

Exported 8 dashboard-ready datasets.


,file,rows,columns,size_kb
0,executive_summary.csv,1,11,0.28
1,monthly_kpis.csv,25,8,1.45
2,category_performance.csv,74,12,6.66
3,distance_performance.csv,6,13,0.72
4,delivery_experience.csv,6,9,0.52
5,customer_retention.csv,2,11,0.38
6,seller_performance.csv,2970,15,332.03
7,state_performance.csv,27,13,2.51
